# Introduction
This notebook serves as a high-level, follow-along guide for anyone who is new to the field of 3D image segmentation, to see how we can implement 3D image segmentation for tagging MRI images. 

If you are a computer science student or otherwise interested in learning more about how to build these models, I recommend you focus on the inline comments for explanations of how the code is accomplishing each task, feel free to also browse the markdown blocks (e.g. this block) to get an idea of what the model is doing in this context and why.

If you are more interested in a high-level understanding of how AI models can be trained and used to accomplish medical imaging tasks, I recommend you focus on the blocks of text you see above the code blocks (like right here)! To understand what each block of code is trying to accomplish and why.


## What is medical image segmentation?
Image segmentation involves training probabilistic models to scan image data, pixel by pixel, and label each pixel according to the structures the model was trained to focus on. Medical image segmentation, specifically, focuses on training models to evaluate medical images such as MRI scans, Xrays, fluoroscopes, and more. Typically these models are trained to identify structures that are harmful to human health. In this project, we will train a model to identify different tumor regions in MRI brain scans. 

## How can a model learn to label tumors?
In this context, a model learns to label tumors by looking at the amount of light (the intensity) reflected by each pixel. The model looks at the intensity, and depending on what it sees, assigns a value to a corresponding pixel in a prediction image. For example, if we are training a model to identify the core of a tumor, surrounding tissue that is damaged by the tumor, and non-tumor areas (background and healthy brain tissue). We may assign values 0, 1, and 2 to each of these categories respectively. Based on the intensity of light reflected within a pixel, the model will categorize that pixel as belonging to group 0, 1, or 2. The model will do this for each pixel in an image, or series of images, and compare its assignments to a reference image that was labeled by medical professionals. And based on the difference between the model's prediction and the true category of each pixel, the model will adjust its future predictions.

For this example, we will use a UNet model architecture to generate our predictions. For more information on what this model is and why it's excellent for such tasks, I recommend you view the following video https://www.youtube.com/watch?v=NhdzGfB1q74


## Where do we begin? 
We begin by setting up the environment for the model to work, this involves loading software that is needed to process data and load model protocols. Below we are loading that software into our current environment.

In [ ]:
'''
First we load all the software libraries required to carry out our segmentation task. No need to read through this list unless you are interested in
reconstructing this repo.
'''
from tqdm import tqdm
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" #allocate sufficient memory for patch-based training
import gc
import tracemalloc
import heapq
import numpy as np #for converting NIfTI images into structured numpy grids
import random
from glob import glob
from pathlib import Path #for navigating to directory with nifti files
import torch
import torchvision
import torchvision.transforms as transforms
from dataset import TrainDataset #for creating training and validation datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt #for visualizing images and plotting summary stats
import nibabel as nib #for loading niftis
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
from DoubleConvolution import UNET
from helpers import ( #functions for previewing patterns in data and planning training loop
    show_sample, #simply visualizes a slice of an mri image
    build_img_set,
    plot_intensity_histograms, 
    estimate_sample_ram_usage, #gives us the memory required for various image patch sizes
    generateSegVoxelCounts, 
    scoreSubjects
)
from transforms import (
    loadimagesd, 
    permutechannels, 
    toTensor, 
    stackTensors, 
    cropToForeground, 
    normalizeIntensities, 
    voxelSpacing, 
    remapLabel, 
    divisiblePad, 
    random_flip, 
    random_rot90, 
    padToShape, 
    centerCropIfLargerThan, 
    classCenteredCrop, 
    SamplePatch
)
from utils import (
    load_checkpoint,
    save_checkpoint,
    check_accuracy_and_loss,
    save_predictions_as_imgs,
    log_batch_loss,
    log_val_metrics,
    get_loaders,
    one_hot_labels,
    soft_dice_loss,
    tversky_loss,
    has_sufficient_voxels,
    is_collapsed_prediction,
    plot_metrics,
    save_overlays,
)

Once our environment is set up, we need to create a map that contains the filepaths of all of our mri images.

Once we have done that, we use show_sample to visualize some images

In [ ]:
#load file paths and store as a dictionary for each subject
#Path makes filepath to data directory, / appends to that filepath
#Training a model requires training data to refine model parameters, and validation data to measure the accuracy of those parameters
data_root = Path("data") / "brats20_output" / "Training" #create path object that points to our data directory
print(type(data_root))

train_subjects, val_subjects = build_img_set(data_root) #split filepaths into training and validation set

all_subjects_list, all_subjects_dict = build_img_set(data_root, asDict = True, split=False)

show_sample(train_subjects, 0, "axial", 119) #visualize one of our samples
show_sample(train_subjects, 0, "coronal", 119)
show_sample(train_subjects, 0, "sagittal", 119)

What we have as the value in each voxel is known as the "intensity". Essentially the amount of light being reflected at that point.
Intensity works a little differently for our different image types. 
T1 scans brighten the fat and darken the water
T2 brightens the water and darkens the fat
FLAIR suppresses CSF so lesions stand out
T1CE T1 with contrast agent, brightens tumors
Intensities can span from 0 to 10000 based on the MRI machine, but our screens can only interpret values 0-255 (8-bit) so we need to optimally scale
the intensities to values that can be interpreted by our machine.

These plots show that intensity can vary a great deal between patients, thus we should scale the intensity on a per-patient basis

In [ ]:
for i in range(10):
    print("subject ", i)
    plot_intensity_histograms(val_subjects[i])

So we have confirmed that we need to adjust the contrast for our images, now we look at the raw image data to glean more insight

In [ ]:
print(nib.load(train_subjects[0]["t1"]).header)
print(nib.load(train_subjects[0]["t1"]).affine)

The rows to pay attention to are dim, pixdim, and datatype dim refers to the dimensions of our 3D image, it is 240 voxels by 240 voxels by 155 voxels, we will crop our images to remove as much background as possible we need to make sure in doing so that the dimensions of each image all match

pixdim refers to the dimensions of each voxel, if the images are not distorted then we would expect these values to be 1 by 1 by 1. The images in this dataset all have non-distorted pixels. Thus we will not need to rescale them during transformation. But normally if we were getting a collection of images from a number of different hospitals, we would expect some images to be distorted when they are rendered on a screen (especially from older MRI machines). It is important to review this and normalize the voxel dimensions for each image

Finally, we see the datatype of our label image is 16 bit integer, this is good and expected, but we need to keep an eye on this. Many transformations we apply will change this datatype to a float... We will need to make sure we change it back to an int

In [ ]:
'''
We use a torch transformation pipeline to apply static and random transforms (augments) to each image in our dataset
static transforms for each image will be applied the same way each time that image is run through the training loop, these transforms include
our numpy arrays to tensors, combining image modalities into a 4D tensor, cropping images, and more

random augments are applied differently each time a given image is run through the training loop. These are typically (but not always) 
lightweight transforms that can be applied quickly. These may include rotating the image, flipping it, extracting pieces of the image. The value of these
augments is that they make it difficult for the model to memorize images where it should be learning patterns.
Since we are doing patch-based training, one of our random augments involves extracting patches from each image
'''
shuffled_subjects = train_subjects[:]
random.shuffle(shuffled_subjects)
transform = transforms.Compose([toTensor, stackTensors, remapLabel, permutechannels, normalizeIntensities,  divisiblePad])
augment = transforms.Compose([random_flip, random_rot90, SamplePatch(patch_size=(96,96,96), percent_random=0.2)])
val_augment = transforms.Compose([toTensor, stackTensors, remapLabel, permutechannels, normalizeIntensities,  divisiblePad, SamplePatch(patch_size= (96,96,96), percent_random=0.2)])

In [ ]:
'''
Here we build the datastructures that will hold our training and validation data. These datasets apply the transformations to each image before it is
used in the training or validation loop.
'''
print("***BUILDING DATASETS*****")
train_dataset = TrainDataset(niftiFiles=train_subjects, transform=transform, augment=augment, cache=True, training=True, max_cache=118)
val_dataset = TrainDataset(niftiFiles=val_subjects, transform=val_transform, augment=None, cache=False, training=False, max_cache=0)

In [ ]:
'''
Some simple hyperparemeters
learning_rate determines the degree to which models weight are adjusted after each batch of images. Too low and our model will not learn, too high and
our model may become unstable and produce nans.
batch_size is the number of images the model will look at before adjusting its weights
num_epochs is the number of training loops the model will run, the way this model is designed, it will rapidly improve over the first 30 or so epochs,
after that, learning will slow immensly 
num_workers refers to the number of batches we operate on in parallel. Only increase this number if you have sufficient ram to hold many images.
pin_memory
Load_model, if true, will start the current training cycle from the model "my_checkpoint.pth.tar" in the current directory
'''
#Hyperparameters
LEARNING_RATE = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 2
NUM_EPOCHS = 1
NUM_WORKERS = 0
PIN_MEMORY = True
LOAD_MODEL = False

In [ ]:
'''
This is our training loop, please peruse the inline comments for any lines that are unclear, but the big picture idea is the following...
we set our model to train so its model weights will be adjusted after each prediction
we tell our model to generate predictions in logit scale
we score our models predictions vs the actual values and adjust model weights
    for first 20 epochs we use cross_entropy alone
    for future epochs we use cross_entropy and dice scores
'''
def train_fn(loader, model, optimizer, loss_ce, scaler, epoch, start_epoch): #scaler
    loop = tqdm(loader)
    model.train()
    loop.set_description(f"Epoch {epoch+start_epoch+1}/{start_epoch+NUM_EPOCHS}")
    c0 = c1 = c2 = c3 = 0
    for batch_idx, batch in enumerate(loop):
        if batch["seg"].max() == 0:
            print("⚠️ Skipping all-background patch")
            continue
        data = batch["image"].to(device=DEVICE)
        targets = batch["seg"].long().to(device=DEVICE)
        if not has_sufficient_voxels(targets):
            print("skipping image with rare class")
            continue
        # Replace any non-finite values
        if not torch.isfinite(data).all():
            data = torch.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
        
        # forward
        with torch.amp.autocast(device_type = "cuda", enabled = False):
            
            predictions = model(data.float()) #cast to float
            with torch.amp.autocast(device_type = "cuda", enabled=False):
                cross_entropy_loss = loss_ce(predictions.float(), targets)

            

            if epoch < 20:
                loss = cross_entropy_loss #use only cross_entropy until model is stable
                print(f"Input device: {data.device}, dtype: {data.dtype}, predictions.dtype{predictions.dtype}  "
                      f"CE: {cross_entropy_loss.item():.4f}")
            else:
                probabilities = torch.softmax(predictions, dim=1)
                probabilities = probabilities.clamp(1e-6, 1 - (1e-6))
                y1h = one_hot_labels(targets, C=4).to(DEVICE, predictions.dtype)
                dice = soft_dice_loss(probabilities, y1h, exclude_bg=True, class_weights=dice_weights)
                loss = cross_entropy_loss + lambda_dice * dice # * tversky
                print(f"Input device: {data.device}, dtype: {data.dtype}, predictions.dtype{predictions.dtype}  "
                        f"CE: {cross_entropy_loss.item():.4f} DiceLoss: {dice.item():.4f} Total: {loss.item():.4f}")
            
            print(f"Prediction shape: {predictions.shape}, Target shape: {targets.shape}")
            print("Logits stats:", predictions.min().item(), predictions.max().item(), predictions.mean().item())
                
            u,c = torch.unique(targets, return_counts=True)
            for cls_id, cnt in zip(u.tolist(), c.tolist()):
                if   cls_id == 0: c0 += cnt
                elif cls_id == 1: c1 += cnt
                elif cls_id == 2: c2 += cnt
                elif cls_id == 3: c3 += cnt

            print(c0, c1, c2, c3)
            gt = dict(zip(u.tolist(), [v.item() for v in c]))
            print("GT: ", gt)
            # print("GT:", dict(zip(u.tolist(), [v.item() for v in c])))
            preds = predictions.detach().argmax(1)
            up,cp = torch.unique(preds, return_counts=True)
            pred = dict(zip(up.tolist(), [v.item() for v in cp]))
            print ("PRED: ", pred)
            # print("PRED:", dict(zip(up.tolist(), [v.item() for v in cp])))
            assert not torch.isnan(predictions).any(), "NaN in model output"
            if torch.isnan(loss).any():
                print("NaN detected in loss! Aborting epoch.")
                break
            loss_val = float(loss.detach())
            log_batch_loss(loss_val, batch_idx, epoch, gt, pred)
        
        #backward
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        total_norm = 0.0
        for name, param in model.named_parameters():
                if param.grad is not None:
                    grad = param.grad.detach()
                    if torch.isnan(grad).any():
                        print(f"❌ NaNs detected in gradients of parameter: {name}")
                        raise ValueError("NaNs in gradients")
                    grad_norm = grad.data.norm(2).item()
                    total_norm += grad_norm ** 2
                    print(f"{name}: grad norm = {grad_norm:.3f}")
        print(f"Total grad norm = {total_norm ** 0.5:.3f}")
        
        for name, param in model.named_parameters():
            if torch.isnan(param.data).any():
                print(f"❌ NaNs detected in model parameter values: {name}")
                raise ValueError("NaNs in model parameters")


        
        optimizer.step()

        #update tqdm loop
        memoryCheck(f"train_b{batch_idx}")
        loop.set_postfix(loss=loss.item())
        if epoch > 20:
            del predictions, probabilities, loss, cross_entropy_loss #y1h, dice
        else:
            del loss, cross_entropy_loss

In [ ]:
model = UNET(in_channels=4, out_channels=4).to(DEVICE) #we load torch's protocol for running unet models
ce_weights = torch.tensor([1.0, 1.0, 1.0, 1.0], device = DEVICE) # we manipulate the rate at which each models weight is adjusted
loss_ce = nn.CrossEntropyLoss(weight=ce_weights)

dice_weights = ce_weights.to(device=DEVICE, dtype=float) #reuse weights for dice emphasis too
lambda_dice = 1.0 # start with 1.0; try 0.5–2.0 if needed
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=0)

In [ ]:
if LOAD_MODEL:
    start_epoch = load_checkpoint(torch.load("my_checkpoint.pth.tar"), model)
else:
    start_epoch = 0
scaler = torch.amp.GradScaler(enabled=False) #could save this, but lets just load it everytime
try:
    for epoch in range(NUM_EPOCHS):
        print(f"\n=== STARTING EPOCH {epoch+1}/{NUM_EPOCHS} ===")
        torch.cuda.reset_peak_memory_stats()
        memoryCheck("epoch_start")
        train_fn(train_loader, model, optimizer, loss_ce, scaler, epoch, start_epoch) #scaler


        #save model
        cpu_sd = {k: v.detach().cpu() for k,v in model.state_dict().items()}
        checkpoint = {
            "epoch" : epoch,
            "state_dict": cpu_sd #model.state_dict(),
            # "optimizer":optimizer.state_dict(),
        }
        if epoch % 5 == 0: #save a checkpoint every 10 epochs
            save_checkpoint(checkpoint)
        # check accuracy and validation loss
        val_loss, mean_dice, class_dice = check_accuracy_and_loss(epoch+start_epoch, val_loader, model, loss_ce, dice_weights, lambda_dice, device=DEVICE)
        memoryCheck("pre_val")
        
        log_val_metrics(start_epoch + epoch, val_loss, mean_dice, class_dice)
        memoryCheck("post_val")
         
        #print some output to a folder
        if epoch == NUM_EPOCHS - 1:
            save_predictions_as_imgs(val_loader, model, folder="saved_images/", device=DEVICE)
            plot_metrics()
            save_overlays(val_loader)
        print(f"\n=== FINISHED EPOCH {epoch+1}/{NUM_EPOCHS} ===")
except Exception as e:
    print(f"[ERROR] Crash at epoch {epoch}: {e}")
    checkpoint = {
            "epoch" : epoch,
            "state_dict": cpu_sd #model.state_dict(),
            # "optimizer":optimizer.state_dict(),
        }
    save_checkpoint(checkpoint, Filename= f"backup_checkpoint_e{epoch}")
    raise
